# ISMIP7 forcing as a DataTree

The ISMIP7 forcing datasets on [source.coop](https://source.coop/ismip) are about
285,000 NetCDF files (6 TB). The [englacial/ismip7](https://source.coop/englacial/ismip7)
Icechunk stores hold *virtual references* into those files (22 GB of metadata, no data
copied). One store per ice sheet, `icechunk-ais` and `icechunk-gris`, laid out as

    <ESM>/<experiment>/<product>/<resolution>m      e.g. CESM2-WACCM/ssp585/SDBN1/2000m

Each resolution group is one xarray Dataset: a normalised monthly (or annual) time axis
in `days since 1850-01-01`, the exact ISMIP7 grid in `x`/`y`, and a `crs` variable with
the EPSG code and WKT. The group above the resolutions (the *family*, e.g.
`CESM2-WACCM/ssp585/SDBN1`) carries an OME-NGFF `multiscales` attribute naming its
levels, so a multiscale-aware reader (gridlook) opens the family as one pyramid.

Reading needs no credentials. Everything below uses the standard tools, `icechunk`,
`zarr` and `xarray`; nothing from `ismip7_helper`.


## Open the repository

Three things a reader must get right: the bucket and prefix, `anonymous=True` (otherwise icechunk looks for AWS credentials), and authorising the virtual-chunk container (wrapped by `containers_credentials`), since icechunk refuses to follow references into a bucket you have not allowed.

In [ ]:
import os
import time

import icechunk
import xarray as xr

REGION = os.environ.get("ISMIP7_NB_REGION", "AIS")  # or "GrIS"
BUCKET = "us-west-2.opendata.source.coop"
PREFIX = f"englacial/ismip7/icechunk-{REGION.lower()}"
# the bucket the virtual chunks point into; icechunk only follows references into
# containers you authorise explicitly
SOURCE = "s3://us-west-2.opendata.source.coop/ismip/"


# the series run to 2300, past datetime64[ns]: decode time with cftime
TIME = xr.coders.CFDatetimeCoder(use_cftime=True)


def read_time(a, tries=3):
    """Seconds to load an xarray object into memory (retrying transient S3 errors)."""
    for attempt in range(tries):
        t0 = time.time()
        try:
            a.load()
            return time.time() - t0
        except Exception:
            if attempt == tries - 1:
                raise
            time.sleep(2)


storage = icechunk.s3_storage(bucket=BUCKET, prefix=PREFIX, region="us-west-2", anonymous=True)
containers = icechunk.containers_credentials({SOURCE: icechunk.s3_anonymous_credentials()})
repo = icechunk.Repository.open(storage, authorize_virtual_chunk_access=containers)
session = repo.readonly_session("main")
print("store:", f"s3://{BUCKET}/{PREFIX}", "| snapshot", session.snapshot_id)

In [ ]:
# rustytree (https://github.com/aladinor/rustytree, `pip install rustytree-xarray`) walks the
# hierarchy concurrently in Rust and reads arrays through its own Rust path, so a whole
# store opens in seconds; the zarr engine does the same work one group at a time
if "rustytree" in xr.backends.list_engines():
    TREE = {"engine": "rustytree"}
else:
    print("rustytree is not installed: using the zarr engine (slower on large trees)")
    TREE = {"engine": "zarr", "consolidated": False}

## The whole store as one tree

Groups are `ESM/experiment/family/resolution`; a family group carries OME-NGFF `multiscales` naming its resolution levels, finest first.

In [ ]:
# the whole store as one lazy tree: metadata only, nothing read until asked
t0 = time.time()
tree = xr.open_datatree(session.store, decode_times=TIME, **TREE)
data = [n for n in tree.subtree if n.data_vars]
families = {
    n.path.strip("/"): [d["path"] for d in n.attrs["multiscales"][0]["datasets"]]
    for n in tree.subtree
    if "multiscales" in n.attrs
}
print(
    f"{REGION}: {len(data)} data groups in {len(families)} families, "
    f"{tree.nbytes / 1e12:.1f} TB virtual, opened in {time.time() - t0:.1f}s"
)
for fam, lv in list(families.items())[:8]:
    print(f"  {fam}: {lv}")

## A family, then one resolution as a Dataset

In [ ]:
# a family: one child per resolution, finest first in `multiscales`
FAMILY = os.environ.get("ISMIP7_NB_FAMILY", "CESM2-WACCM/historical/SDBN1")
family = tree[FAMILY]
print(family.attrs["multiscales"][0]["datasets"])
print({child: dict(family[child].ds.sizes) for child in family.children})

In [ ]:
# one resolution as a Dataset (the coarsest level of the family)
GROUP = f"{FAMILY}/{families[FAMILY][-1]}"
ds = tree[GROUP].to_dataset()
ds

In [ ]:
var = next(v for v in ds.data_vars if ds[v].ndim >= 3)
tdim = ds[var].dims[0]  # `time`, or `time_annual` for the annual products
print(tdim, ds[tdim].values[0], "->", ds[tdim].values[-1], f"({ds.sizes[tdim]} steps)")
print("grid:", ds.attrs.get("ismip7_grid"), "| crs:", ds.crs.attrs.get("epsg_code"))
print(var, "chunks", ds[var].encoding.get("chunks"))
print("actual_range", ds[var].attrs.get("actual_range"))
for n in ds.attrs.get("ismip7_notes", [])[:5]:
    print(" note:", n)

## Plot one month

Only the chunk(s) of that time step are fetched from the source file; the fill value is masked to NaN.

In [ ]:
import numpy as np

field = ds[var].isel({tdim: 0})
secs = read_time(field)
print(f"{var} t=0 read in {secs:.2f}s; finite cells {int(np.isfinite(field).sum())}")
field.plot(x="x", y="y", robust=True, figsize=(6, 6));

## Subset and reduce

A spatial window and a mean over the first year; everything stays lazy until it is loaded.

In [ ]:
n = min(12, ds.sizes[tdim])
box = ds[var].isel({tdim: slice(0, n), "x": slice(100, 300), "y": slice(100, 300)})
secs = read_time(box)
print(f"{n} steps x 200 x 200 cells read in {secs:.2f}s")
box.mean(tdim).plot(x="x", y="y", robust=True, figsize=(5, 5));

## Both ice sheets in one tree

In [ ]:
# both ice sheets as one tree: AIS and GrIS are separate Icechunk repositories, so open
# each and nest them
trees = {}
for region in ("AIS", "GrIS"):
    prefix = f"englacial/ismip7/icechunk-{region.lower()}"
    stor = icechunk.s3_storage(bucket=BUCKET, prefix=prefix, region="us-west-2", anonymous=True)
    sess = icechunk.Repository.open(stor, authorize_virtual_chunk_access=containers)
    trees[region] = xr.open_datatree(sess.readonly_session("main").store, decode_times=TIME, **TREE)
both = xr.DataTree.from_dict(trees)
n_data = {r: sum(1 for n in t.subtree if n.data_vars) for r, t in trees.items()}
print(sum(1 for _ in both.subtree), "nodes;", n_data)
print(both["GrIS/CESM2-WACCM/historical/SDBN1"].attrs["multiscales"][0]["datasets"])

## Scope a walk with a glob

In [ ]:
# scope a walk with a glob over full group paths (leading slash): e.g. every historical
# 8 km group of this store (rustytree only; the zarr engine ignores group_filter)
if TREE["engine"] == "rustytree":
    t0 = time.time()
    hist8 = xr.open_datatree(
        session.store, group_filter="/*/historical/*/8000m", decode_times=TIME, **TREE
    )
    print([n.path for n in hist8.subtree if n.data_vars], f"in {time.time() - t0:.1f}s")